# Day 2: ทำความสะอาดและรวมข้อมูล (Cleaning)

ต่อจาก `01_data_check.ipynb` โดยเอาข้อมูลดิบ `openaq_hourly_raw.csv` มา:
1. ติดธงข้อมูลน่าสงสัย (`is_suspect_stuck`)
2. เช็ก/แปลง timezone เป็น Asia/Bangkok
3. Pivot เป็น wide format
4. Merge กับข้อมูลลมจาก Open-Meteo

**หมายเหตุ:** notebook นี้รันใหม่ใน Day 4 หลังเพิ่มสถานีอ้างอิงสำรอง 225690 Agricultural Office (role `background_alt`) เพราะ Day 4 พบว่าสถานีอ้างอิง 717 มีพื้นที่อุตสาหกรรมรอบตัวมาก ผลของ 3 สถานีเดิมไม่เปลี่ยน

## 1. โหลดข้อมูลดิบ

In [1]:
import pandas as pd
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/raw/openaq_hourly_raw.csv', parse_dates=['datetime_utc','datetime_local'])
print('จำนวนแถวทั้งหมด:', len(df))
display(df[['station_id','station_name','role','parameter','datetime_utc','value']].head(3))


จำนวนแถวทั้งหมด: 510697


,station_id,station_name,role,parameter,datetime_utc,value
0,225611,Health Promotion Hospital Maptaput,near,co,2021-04-29 17:00:00+00:00,0.37
1,225611,Health Promotion Hospital Maptaput,near,co,2021-04-29 19:00:00+00:00,0.34
2,225611,Health Promotion Hospital Maptaput,near,co,2021-04-29 21:00:00+00:00,0.37


## 2. ติดธง `is_suspect_stuck`

ใช้เกณฑ์ที่ตกลงกันไว้จาก `01_data_check.ipynb`:
- ค่า **ไม่เป็น 0** ซ้ำติดกัน **≥ 24 ชั่วโมง** → น่าสงสัย
- ค่า **เป็น 0** ซ้ำติดกัน **≥ 72 ชั่วโมง** → น่าสงสัย (ผ่อนปรนกว่า เพราะอาจเป็น below-detection-limit จริง)

เป็นการ "ติดป้าย" ไม่ใช่ "ลบทิ้ง" ดังนั้นข้อมูลเดิมยังอยู่ครบ

In [2]:
df = df.sort_values(['station_id','parameter','datetime_utc']).reset_index(drop=True)

ZERO_MIN, NONZERO_MIN = 72, 24
flags = []
for (sid, p), g in df.groupby(['station_id','parameter']):
    g = g.sort_values('datetime_utc')
    run_id = (~g['value'].eq(g['value'].shift())).cumsum()
    n = g.groupby(run_id)['value'].transform('size')
    is_zero = g['value'].eq(0)
    flag = ((is_zero) & (n >= ZERO_MIN)) | ((~is_zero) & (n >= NONZERO_MIN))
    flags.append(pd.Series(flag.values, index=g.index))

df['is_suspect_stuck'] = pd.concat(flags).sort_index()

print('ตรวจสอบ: จำนวนที่ติดธงรวม =', df['is_suspect_stuck'].sum())
print('% รวม =', round(100 * df['is_suspect_stuck'].mean(), 2))
print('แยกตามสถานี:', df.groupby('station_id')['is_suspect_stuck'].sum().to_dict())
print()
display(df[['station_id','parameter','datetime_utc','value','is_suspect_stuck']].head(3))


ตรวจสอบ: จำนวนที่ติดธงรวม = 7896
% รวม = 1.55
แยกตามสถานี: {717: 4654, 7846: 30, 225611: 673, 225690: 2539}



,station_id,parameter,datetime_utc,value,is_suspect_stuck
0,717,no2,2016-01-30 00:00:00+00:00,0.013,False
1,717,no2,2016-01-30 01:00:00+00:00,0.009,False
2,717,no2,2016-01-30 02:00:00+00:00,0.016,False


ติดธงรวม 7,896 แถว (1.55%)

**ข้อสังเกต:** SO₂ ของ 225690 ถูกติดธงถึง 24.7% (1,991 แถว) ค่าที่ซ้ำคือ 0.001 ถึง 0.003 ppm ซึ่งเป็นความละเอียดต่ำสุดของข้อมูล จึงอาจเป็นค่าต่ำจริงที่ซ้ำกันเอง ไม่ใช่เซนเซอร์ค้าง (ยังไม่ได้พิสูจน์) ไม่ได้แก้เกณฑ์ เพราะ Day 3 และ Day 4 ไม่ได้ใช้ธงนี้

## 3. เช็ก timezone

เช็กว่า `datetime_local` ที่ OpenAQ API ให้มา ตรงกับการแปลงจาก UTC → Asia/Bangkok ด้วยมือเองหรือไม่

In [3]:
print('dtype ของ datetime_utc:', df['datetime_utc'].dtype)
print('dtype ของ datetime_local (จาก API):', df['datetime_local'].dtype)
print()

df['datetime_bkk'] = df['datetime_utc'].dt.tz_convert('Asia/Bangkok')
print('ตัวอย่างหลังแปลงด้วยมือเองจาก UTC -> Asia/Bangkok เทียบกับที่ API ให้มา:')
display(df[['datetime_utc','datetime_local','datetime_bkk']].head(3))

mismatch = (df['datetime_bkk'] != df['datetime_local']).sum()
print(f'จำนวนแถวที่ datetime_local (จาก API) กับที่เราแปลงเองไม่ตรงกัน: {mismatch} จาก {len(df)} แถว')


dtype ของ datetime_utc: datetime64[us, UTC]
dtype ของ datetime_local (จาก API): datetime64[us, UTC+07:00]

ตัวอย่างหลังแปลงด้วยมือเองจาก UTC -> Asia/Bangkok เทียบกับที่ API ให้มา:


,datetime_utc,datetime_local,datetime_bkk
0,2016-01-30 00:00:00+00:00,2016-01-30 07:00:00+07:00,2016-01-30 07:00:00+07:00
1,2016-01-30 01:00:00+00:00,2016-01-30 08:00:00+07:00,2016-01-30 08:00:00+07:00
2,2016-01-30 02:00:00+00:00,2016-01-30 09:00:00+07:00,2016-01-30 09:00:00+07:00


จำนวนแถวที่ datetime_local (จาก API) กับที่เราแปลงเองไม่ตรงกัน: 0 จาก 510697 แถว


**สรุป:** ตรงกัน 100% (mismatch = 0) เพราะไทยใช้ UTC+7 คงที่ตลอดปี ไม่มี Daylight Saving Time ใช้คอลัมน์ `datetime_local` เดิมจาก API ได้เลย ไม่ต้องแปลงเพิ่ม

## 4. Pivot เป็น wide format

เปลี่ยนจาก **1 แถว = 1 สถานี + 1 สาร + 1 ชั่วโมง** ให้เป็น **1 แถว = 1 สถานี + 1 ชั่วโมง** โดยแต่ละสาร (so2, no2, pm25, ...) กลายเป็นคอลัมน์แยก เพื่อให้คำนวณ "local increment" (ค่าใกล้นิคม − ค่าสถานีอ้างอิง ที่เวลาเดียวกัน) ใน Day 3 ได้ง่าย

ธง `is_suspect_stuck` ก็ pivot คู่กันไปด้วย เป็นคอลัมน์ `<สาร>_flag`

In [4]:
wide_val = df.pivot_table(
    index=['station_id','station_name','role','datetime_utc','datetime_local'],
    columns='parameter', values='value', aggfunc='first'
)
wide_flag = df.pivot_table(
    index=['station_id','station_name','role','datetime_utc','datetime_local'],
    columns='parameter', values='is_suspect_stuck', aggfunc='first'
)
wide_flag.columns = [f'{c}_flag' for c in wide_flag.columns]

wide = wide_val.join(wide_flag).reset_index()
print('shape ของ wide format:', wide.shape)
print()
print('คอลัมน์ทั้งหมด:', list(wide.columns))
print()
display(wide[wide.station_id==225611].head(3))


shape ของ wide format: (151093, 17)

คอลัมน์ทั้งหมด: ['station_id', 'station_name', 'role', 'datetime_utc', 'datetime_local', 'co', 'no2', 'o3', 'pm10', 'pm25', 'so2', 'co_flag', 'no2_flag', 'o3_flag', 'pm10_flag', 'pm25_flag', 'so2_flag']



,station_id,station_name,role,datetime_utc,datetime_local,co,no2,o3,pm10,pm25,so2,co_flag,no2_flag,o3_flag,pm10_flag,pm25_flag,so2_flag
98882,225611,Health Promotion Hospital Maptaput,near,2021-04-29 17:00:00+00:00,2021-04-30 00:00:00+07:00,0.37,0.013,0.026,24.0,NaN,0.003,False,False,False,False,NaN,False
98883,225611,Health Promotion Hospital Maptaput,near,2021-04-29 19:00:00+00:00,2021-04-30 02:00:00+07:00,0.34,0.008,0.035,24.0,NaN,0.002,False,False,False,False,NaN,False
98884,225611,Health Promotion Hospital Maptaput,near,2021-04-29 21:00:00+00:00,2021-04-30 04:00:00+07:00,0.37,0.011,0.034,26.0,NaN,0.001,False,False,False,False,NaN,False


**ข้อสังเกต:** บางช่องเป็น `NaN` เช่น `pm25` ปลายเดือนเมษายน 2021 เพราะเซนเซอร์ PM2.5 ของสถานีนี้เริ่มวัดต้นเดือนพฤษภาคม ไม่ใช่ error

In [5]:
print('เช็กจำนวนแถวให้ตรงกับ unique station+hour ของข้อมูลดิบ:')
check = df.groupby('station_id')['datetime_utc'].nunique()
print(check)
print('รวม:', check.sum(), '  ตรงกับ wide.shape[0] หรือไม่:', check.sum() == wide.shape[0])


เช็กจำนวนแถวให้ตรงกับ unique station+hour ของข้อมูลดิบ:
station_id
717       61791
7846      37091
225611    29761
225690    22450
Name: datetime_utc, dtype: int64
รวม: 151093   ตรงกับ wide.shape[0] หรือไม่: True


**ยืนยัน:** 151,093 แถว ตรงกับจำนวนชั่วโมงไม่ซ้ำของ 4 สถานีรวมกัน ไม่มีแถวหายหรือเกินจากการ pivot

## 5. โหลดข้อมูลลมจาก Open-Meteo

ดึงไว้แล้วด้วย `src/fetch_wind.py` โดยใช้ 2 โมเดล:

| โมเดล | grid | บทบาท |
|---|---|---|
| **ECMWF IFS** | 9 กม. | **ตัวหลัก** ที่แยกสถานีใกล้นิคมทั้งสองได้จริง (ทดสอบแล้วทิศลมต่างกัน 95.8% ของชั่วโมง) |
| ERA5 | 25 กม. | ตัวสำรองไว้ทำ sensitivity check ซึ่งย้อนหลังได้ถึง 2016 และสม่ำเสมอข้ามปี |

**เหตุผลที่ไม่ใช้ ERA5 เป็นตัวหลัก:** สถานีใกล้นิคมทั้งสองห่างกันเพียง 4.42 กม. ซึ่งเล็กกว่า grid 25 กม. ของ ERA5 → ตกใน grid cell เดียวกัน ได้ค่าลมเหมือนกัน (ทดสอบแล้วต่างกันเพียง 144 จาก 94,272 ชม. = 0.15%)

**ตัวแปรที่ตัดออก:** `boundary_layer_height` เพราะ API คืนค่าแค่ 7 ชม.แรกของวัน และค่าที่ได้ (20-50 ม.) ต่ำผิดปกติจนไม่น่าเชื่อถือ

In [6]:
wind = pd.read_csv('../data/raw/wind_hourly_raw.csv', parse_dates=['datetime_utc'])
print('shape ของข้อมูลลมดิบ:', wind.shape)
print('timezone ของ datetime_utc:', wind['datetime_utc'].dt.tz)

# Open-Meteo ขอมาเป็น UTC แต่ไม่ได้ใส่ offset มาในสตริง ต้อง localize ก่อน merge
wind['datetime_utc'] = wind['datetime_utc'].dt.tz_localize('UTC')
print('หลัง localize เป็น UTC:', wind['datetime_utc'].dt.tz)
print()
print('จำนวนชั่วโมงที่มีทิศลมจริง แยกตามโมเดล:')
display(wind.groupby('wind_model')['wind_direction_10m'].agg(['size','count']))


shape ของข้อมูลลมดิบ: (754272, 10)
timezone ของ datetime_utc: None
หลัง localize เป็น UTC: UTC

จำนวนชั่วโมงที่มีทิศลมจริง แยกตามโมเดล:


,size,count
wind_model,,
ecmwf_ifs,377136,342092
era5,377136,376560


**ข้อสังเกต:** IFS มีทิศลม 90.7% ของชั่วโมง ที่ขาดคือปี 2016 ถึงต้นปี 2017 ส่วน ERA5 มี 99.8% ลมของ 225690 มีมากกว่าสถานีอื่น 48 ชั่วโมง เพราะดึงทีหลัง 2 วัน ไม่กระทบผลเพราะ merge แบบ left join จากตารางอากาศ

## 6. Merge ลมเข้ากับตารางอากาศ

merge ด้วย `station_id` + `datetime_utc` แบบ left join (เก็บทุกชั่วโมงที่มีข้อมูลอากาศไว้ ถึงลมจะขาดก็ตาม) แยกคอลัมน์ของสองโมเดลด้วย prefix `ifs_` และ `era5_`

In [7]:
WIND_COLS = ['wind_speed_10m','wind_direction_10m','wind_gusts_10m',
             'precipitation','temperature_2m','relative_humidity_2m']

merged = wide.copy()
for model, prefix in [('ecmwf_ifs','ifs'), ('era5','era5')]:
    sub = wind[wind.wind_model == model][['station_id','datetime_utc'] + WIND_COLS].copy()
    sub = sub.rename(columns={c: f'{prefix}_{c}' for c in WIND_COLS})
    merged = merged.merge(sub, on=['station_id','datetime_utc'], how='left')

print('shape หลัง merge:', merged.shape)
print('(ก่อน merge wide มี', wide.shape[0], 'แถว -> ต้องเท่ากัน ไม่งอกแถว)')
print()
print('คอลัมน์ลมที่เพิ่มมา:', [c for c in merged.columns if c.startswith(('ifs_','era5_'))])


shape หลัง merge: (151093, 29)
(ก่อน merge wide มี 151093 แถว -> ต้องเท่ากัน ไม่งอกแถว)

คอลัมน์ลมที่เพิ่มมา: ['ifs_wind_speed_10m', 'ifs_wind_direction_10m', 'ifs_wind_gusts_10m', 'ifs_precipitation', 'ifs_temperature_2m', 'ifs_relative_humidity_2m', 'era5_wind_speed_10m', 'era5_wind_direction_10m', 'era5_wind_gusts_10m', 'era5_precipitation', 'era5_temperature_2m', 'era5_relative_humidity_2m']


**ยืนยัน:** หลัง merge ยังเป็น 151,093 แถว จับคู่ 1:1 ไม่มีลมซ้ำ

In [8]:
print('% ชั่วโมงที่ merge ลมติด (มีทิศลม) แยกตามสถานี')
cov = merged.groupby(['station_id','role']).agg(
    ชั่วโมงทั้งหมด=('datetime_utc','size'),
    ifs_มีลม=('ifs_wind_direction_10m','count'),
    era5_มีลม=('era5_wind_direction_10m','count'),
)
cov['ifs_%'] = (100*cov['ifs_มีลม']/cov['ชั่วโมงทั้งหมด']).round(1)
cov['era5_%'] = (100*cov['era5_มีลม']/cov['ชั่วโมงทั้งหมด']).round(1)
display(cov)


% ชั่วโมงที่ merge ลมติด (มีทิศลม) แยกตามสถานี


,,ชั่วโมงทั้งหมด,ifs_มีลม,era5_มีลม,ifs_%,era5_%
station_id,role,,,,,
717,background,61791,56064,61664,90.7,99.8
7846,near,37091,34755,37091,93.7,100.0
225611,near,29761,29761,29693,100.0,99.8
225690,background_alt,22450,22450,22450,100.0,100.0


สถานีใกล้นิคม 225611 (ตัวหลักของโปรเจกต์) ได้ลม IFS ครบ **100%** เพราะข้อมูลอากาศเริ่มปี 2021 ซึ่งอยู่ในช่วงที่ IFS มีข้อมูลเต็ม สถานีอ้างอิงสำรอง 225690 ก็ได้ครบ 100% ทั้งสองโมเดลด้วยเหตุผลเดียวกัน

ส่วน 717 และ 7846 ที่มีข้อมูลอากาศย้อนไปถึงปี 2016 จึงขาดลม IFS บางช่วง (90.7% / 93.7%) แต่ช่วงนั้นยังใช้ ERA5 ได้ (99.8% / 100%)

In [9]:
print('ตัวอย่างข้อมูลหลัง merge (สถานีใกล้นิคม 225611)')
cols = ['station_id','datetime_local','so2','pm25',
        'ifs_wind_direction_10m','ifs_wind_speed_10m',
        'era5_wind_direction_10m','era5_wind_speed_10m']
display(merged[merged.station_id==225611][cols].head(5))
print('เทียบทิศลม IFS vs ERA5 ของ 2 สถานีใกล้นิคม ที่ชั่วโมงเดียวกัน')
t = merged[merged.datetime_utc=='2024-01-01 01:00:00+00:00'][
    ['station_id','role','ifs_wind_direction_10m','era5_wind_direction_10m']]
display(t)

merged.to_parquet('../data/processed/hourly_merged.parquet', index=False)
print()
print('บันทึก data/processed/hourly_merged.parquet แล้ว', merged.shape)


ตัวอย่างข้อมูลหลัง merge (สถานีใกล้นิคม 225611)


,station_id,datetime_local,so2,pm25,ifs_wind_direction_10m,ifs_wind_speed_10m,era5_wind_direction_10m,era5_wind_speed_10m
98882,225611,2021-04-30 00:00:00+07:00,0.003,NaN,36.0,4.9,62.0,5.3
98883,225611,2021-04-30 02:00:00+07:00,0.002,NaN,45.0,4.1,29.0,3.7
98884,225611,2021-04-30 04:00:00+07:00,0.001,NaN,36.0,3.1,301.0,2.1
98885,225611,2021-04-30 05:00:00+07:00,0.001,NaN,27.0,4.8,349.0,1.8
98886,225611,2021-04-30 06:00:00+07:00,0.000,NaN,31.0,4.2,5.0,4.0


เทียบทิศลม IFS vs ERA5 ของ 2 สถานีใกล้นิคม ที่ชั่วโมงเดียวกัน


,station_id,role,ifs_wind_direction_10m,era5_wind_direction_10m
44088,717,background,28.0,42.0
81826,7846,near,32.0,35.0
114126,225611,near,24.0,35.0
144652,225690,background_alt,10.0,35.0



บันทึก data/processed/hourly_merged.parquet แล้ว (151093, 29)


## สรุปผล Day 2

ได้ `data/processed/hourly_merged.parquet` ขนาด 151,093 แถว × 29 คอลัมน์ โดย 1 แถวคือ 1 สถานี ต่อ 1 ชั่วโมง มีธง `is_suspect_stuck` และทิศลมจาก 2 โมเดล